Loop through the CMIP6 model library and save a dictionary of models for analysis in the T-S plot (Figure 4) and the temperature maps Supporting Figure S1.

In [1]:
import matplotlib.pyplot as plt
import intake
import dask
import intake
import numpy as np
import xarray as xr
import cf_xarray as cfxr
%matplotlib inline

In [2]:
#Access the CMIP6 cloud storage from pangeo
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)

In [5]:
from xmip.preprocessing import combined_preprocessing

query = dict(
    experiment_id=['historical'],
    table_id='Omon', 
    member_id='r1i1p1f1',# ← Only one ensemble member!
    variable_id=['thetao', 'so'],
    grid_label='gn',  # ← Only native grid
)

cat = col.search(**query)
print(f"Found {len(cat.df)} datasets for {cat.df['source_id'].nunique()} models")

z_kwargs = {'consolidated': True, 'decode_times': False}  # argument to consolidate metadata, and keep native times

with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    dset_dict = cat.to_dataset_dict(
        zarr_kwargs=z_kwargs,                    # Zarr read options set above
        preprocess=combined_preprocessing,       # xmip preprocessing
        progressbar=True,                        # Show loading progress
        skip_on_error=True                       # Skip problematic models
    )
#after running above, dset_dict is the output, whcih is a dictionary containing all the xarray model names and coordinates (NOT DATA)

Found 85 datasets for 44 models

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


/Users/jaimepalter/opt/anaconda3/lib/python3.9/site-packages/xmip/preprocessing.py:229: UserWarning: CMIP.AWI.AWI-ESM-1-1-LR.historical.r1i1p1f1.Omon.gn.none.so: Unit correction failed with: Cannot convert variables:
    incompatible units for variable 'lev': cannot convert a non-quantity using 'm' as unit
  warnings.warn(
/Users/jaimepalter/opt/anaconda3/lib/python3.9/site-packages/xmip/preprocessing.py:229: UserWarning: CMIP.IPSL.IPSL-CM6A-LR-INCA.historical.r1i1p1f1.Omon.gn.none.so: Unit correction failed with: Cannot convert variables:
    incompatible units for variable 'lev': cannot convert a non-quantity using 'm' as unit
  warnings.warn(
/Users/jaimepalter/opt/anaconda3/lib/python3.9/site-packages/xmip/preprocessing.py:229: UserWarning: CMIP.AWI.AWI-ESM-1-1-LR.historical.r1i1p1f1.Omon.gn.none.thetao: Unit correction failed with: Cannot convert variables:
    incompatible units for variable 'lev': cannot convert a non-quantity using 'm' as unit
  warnings.warn(
/Users/jaimepalte

In [7]:

# Save dataset dictionary (so I don't have to run the previous line every time)
import pickle
from pathlib import Path

save_path = Path('/Users/jaimepalter/OneDrive - University of Rhode Island/New Ideas/AMOC Lunch Bunch/dset_dict.pkl')

# Now save
with open(save_path, 'wb') as f:
    pickle.dump(dset_dict, f)
    print('Already saved, do not replace')

# Later, load it using:
#with open('dset_dict.pkl', 'rb') as f:
#    dset_dict = pickle.load(f)

Already saved, do not replace
